This notebook covers data wrangling techniques using **Polars**, a fast DataFrame library built on Apache Arrow.

We work with two datasets from the `data/` folder:
- `ingatlan-listings.parquet` — 9M+ price observations per property per day  
- `ingatlan-details.parquet` — property-level attributes

`ingatlan-listings.parquet`

| Column | Type | Description |
|--------|------|-------------|
| `id` | `i64` | Unique property identifier |
| `day` | `datetime[μs]` | Observation date (when the price was recorded) |
| `price` | `i64` | Listed price in thousands of HUF |

9M+ rows — one row per property per day it was listed.

`ingatlan-details.parquet`

| Column | Type | Description |
|--------|------|-------------|
| `id` | `i64` | Unique property identifier (joins to `listings.id`) |
| `loc` | `str` | Full location string, e.g. `"IX. kerület, Lónyay utca "` |
| `city` | `str` | City name, e.g. `"Budapest"`, `"Debrecen"` |
| `price` | `i64` | Reference price at time of scraping (thousands of HUF) |
| `m2` | `i64` | Property area in square metres |
| `rooms` | `str` | Room count as free text, e.g. `"3"`, `"1 fél"`, `"2 + 1 fél"` |
| `balcony` | `str` | Balcony area as free text, e.g. `"10 m2"`, `""`, or `null` |

221K rows — one row per unique property.

In [20]:
import polars as pl

In [21]:
listings_df = pl.read_parquet("data/ingatlan-listings.parquet")
listings_df.head()

id,day,price
i64,datetime[μs],i64
576781,2025-06-21 00:00:00,362466
576781,2025-05-22 00:00:00,362340
576781,2025-07-31 00:00:00,358893
576781,2025-06-04 00:00:00,362934
576781,2025-07-01 00:00:00,359370


In [22]:
details_df = pl.read_parquet("data/ingatlan-details.parquet")
details_df.head()

loc,city,price,m2,rooms,balcony,id
str,str,i64,i64,str,str,i64
"""IX. kerület, Lónyay utca ""","""Budapest""",498862,95,"""3""","""""",7065343
"""Gárdony, Agárd ""","""Gárdony""",120000,25,"""1 fél""","""""",33544237
"""VI. kerület, Hajós utca ""","""Budapest""",306992,41,"""1""","""""",32889588
"""II. kerület, Török utca ""","""Budapest""",435000,125,"""3""","""10 m2""",33717225
"""V. kerület, Vörösmarty tér ""","""Budapest""",1290000,205,"""6""","""""",33611258


# Pandas vs Polars

In [23]:
import pandas as pd

details_pdf = pd.read_parquet("data/ingatlan-details.parquet")
details_pdf.head()

,loc,city,price,m2,rooms,balcony,id
0,"IX. kerület, Lónyay utca",Budapest,498862,95,3,,7065343
1,"Gárdony, Agárd",Gárdony,120000,25,1 fél,,33544237
2,"VI. kerület, Hajós utca",Budapest,306992,41,1,,32889588
3,"II. kerület, Török utca",Budapest,435000,125,3,10 m2,33717225
4,"V. kerület, Vörösmarty tér",Budapest,1290000,205,6,,33611258


In [24]:
details_pdf["city"]

0         Budapest
1          Gárdony
2         Budapest
3         Budapest
4         Budapest
            ...   
221073    Budapest
221074    Budapest
221075    Budapest
221076        Tata
221077        Tata
Name: city, Length: 221078, dtype: str

In [25]:
details_pdf.loc[1, 'city']

'Gárdony'

In [26]:
details_df.select("city")

city
str
"""Budapest"""
"""Gárdony"""
"""Budapest"""
"""Budapest"""
"""Budapest"""
"""Budapest"""
"""Balatonfüred"""
"""Budapest"""
"""Budapest"""


In [27]:
details_df.select(
    pl.col("city").filter(pl.row_index() == 1),
)

city
str
"""Gárdony"""


In [28]:
details_pdf.assign(
    city_avg=lambda _df: (
        _df
        .groupby("city")["price"]
        .transform("mean")
    )
)

,loc,city,price,m2,rooms,balcony,id,city_avg
0,"IX. kerület, Lónyay utca",Budapest,498862,95,3,,7065343,1.454661e+06
1,"Gárdony, Agárd",Gárdony,120000,25,1 fél,,33544237,1.696098e+05
2,"VI. kerület, Hajós utca",Budapest,306992,41,1,,32889588,1.454661e+06
3,"II. kerület, Török utca",Budapest,435000,125,3,10 m2,33717225,1.454661e+06
4,"V. kerület, Vörösmarty tér",Budapest,1290000,205,6,,33611258,1.454661e+06
...,...,...,...,...,...,...,...,...
221073,"Budapest VI. kerület, Külső-Terézváros",Budapest,400000,140,3,NaN,35234253,1.454661e+06
221074,"Budapest XII. kerület, Kiss János altábornagy ...",Budapest,195000,56,2,NaN,35234011,1.454661e+06
221075,"Budapest VIII. kerület, Corvin sétány 2.",Budapest,490000000,70,2 + 1 fél,8 m2,35232596,1.454661e+06
221076,"Tata, Deák Ferenc utca",Tata,310000,93,4,15.01 m2,35236629,1.318495e+06


In [29]:
details_df.with_columns(
    pl.col("price").mean().over("city").alias("city_avg"),
)

loc,city,price,m2,rooms,balcony,id,city_avg
str,str,i64,i64,str,str,i64,f64
"""IX. kerület, Lónyay utca ""","""Budapest""",498862,95,"""3""","""""",7065343,1.4547e6
"""Gárdony, Agárd ""","""Gárdony""",120000,25,"""1 fél""","""""",33544237,169609.756098
"""VI. kerület, Hajós utca ""","""Budapest""",306992,41,"""1""","""""",32889588,1.4547e6
"""II. kerület, Török utca ""","""Budapest""",435000,125,"""3""","""10 m2""",33717225,1.4547e6
"""V. kerület, Vörösmarty tér ""","""Budapest""",1290000,205,"""6""","""""",33611258,1.4547e6
"""XII. kerület, Kissvábhegy ""","""Budapest""",270000,46,"""1 + 1 fél""","""23 m2""",33339351,1.4547e6
"""Balatonfüred, Fürdőtelep ""","""Balatonfüred""",140000,30,"""1""","""""",33698882,594153.898577
"""IX. kerület, Lechner Ödön faso…","""Budapest""",575610,78,"""3""","""15 m2""",33588947,1.4547e6
"""VI. kerület, Paulay Ede utca ""","""Budapest""",441301,78,"""3""","""5 m2""",23928384,1.4547e6


In [30]:
details_df.with_columns(
    city_avg = pl.col("price").filter(pl.col("m2") > 100).mean().over("city"),
)

loc,city,price,m2,rooms,balcony,id,city_avg
str,str,i64,i64,str,str,i64,f64
"""IX. kerület, Lónyay utca ""","""Budapest""",498862,95,"""3""","""""",7065343,4.4684e6
"""Gárdony, Agárd ""","""Gárdony""",120000,25,"""1 fél""","""""",33544237,null
"""VI. kerület, Hajós utca ""","""Budapest""",306992,41,"""1""","""""",32889588,4.4684e6
"""II. kerület, Török utca ""","""Budapest""",435000,125,"""3""","""10 m2""",33717225,4.4684e6
"""V. kerület, Vörösmarty tér ""","""Budapest""",1290000,205,"""6""","""""",33611258,4.4684e6
"""XII. kerület, Kissvábhegy ""","""Budapest""",270000,46,"""1 + 1 fél""","""23 m2""",33339351,4.4684e6
"""Balatonfüred, Fürdőtelep ""","""Balatonfüred""",140000,30,"""1""","""""",33698882,623138.75
"""IX. kerület, Lechner Ödön faso…","""Budapest""",575610,78,"""3""","""15 m2""",33588947,4.4684e6
"""VI. kerület, Paulay Ede utca ""","""Budapest""",441301,78,"""3""","""5 m2""",23928384,4.4684e6


In [31]:
details_pdf.pipe(
    lambda _df: _df.merge(
        (
            _df.loc[lambda _fdf: _fdf["m2"] > 100, :]
            .groupby("city")["price"]
            .mean()
            .rename("city_avg")
        ),
        left_on="city",
        right_index=True,
        how="left",
    ),
)


,loc,city,price,m2,rooms,balcony,id,city_avg
0,"IX. kerület, Lónyay utca",Budapest,498862,95,3,,7065343,4.468401e+06
1,"Gárdony, Agárd",Gárdony,120000,25,1 fél,,33544237,NaN
2,"VI. kerület, Hajós utca",Budapest,306992,41,1,,32889588,4.468401e+06
3,"II. kerület, Török utca",Budapest,435000,125,3,10 m2,33717225,4.468401e+06
4,"V. kerület, Vörösmarty tér",Budapest,1290000,205,6,,33611258,4.468401e+06
...,...,...,...,...,...,...,...,...
221073,"Budapest VI. kerület, Külső-Terézváros",Budapest,400000,140,3,NaN,35234253,4.468401e+06
221074,"Budapest XII. kerület, Kiss János altábornagy ...",Budapest,195000,56,2,NaN,35234011,4.468401e+06
221075,"Budapest VIII. kerület, Corvin sétány 2.",Budapest,490000000,70,2 + 1 fél,8 m2,35232596,4.468401e+06
221076,"Tata, Deák Ferenc utca",Tata,310000,93,4,15.01 m2,35236629,6.393602e+06


# Query optimization

----------

## Simple solution

In [32]:
listings_df = listings_df.sample(fraction=1.0, seed=42)

In [33]:
pl.Config.set_tbl_rows(25)

polars.config.Config

In [34]:
def low_iqr_props(
    details: pl.DataFrame, listings: pl.DataFrame, top_k: int = 20
) -> pl.DataFrame:
    return details.join(
        (
            listings.group_by("id")
            .agg(
                iqr=(pl.col("price").quantile(0.75) - pl.col("price").quantile(0.25)),
            )
            .with_columns(
                pl.col("iqr").rank(method="min", descending=True).alias("rank"),
            )
            .filter(pl.col("rank") <= top_k)
        ),
        on="id",
        how="semi",
    )

In [35]:
%%timeit -r 10 -n 1
low_iqr_props(details_df, listings_df)

1.31 s ± 190 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [36]:
listings_df = listings_df.sort(
    "id",
    "price",
)

In [37]:
%%timeit -r 10 -n 1
low_iqr_props(details_df, listings_df)

565 ms ± 55.1 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [38]:
listings_df.write_parquet("data/ingatlan-listings-sorted.parquet")
listings_df = pl.read_parquet("data/ingatlan-listings-sorted.parquet")

In [39]:
%%timeit -r 10 -n 1
low_iqr_props(details_df, listings_df)

1.01 s ± 71.8 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [40]:
listings_df.write_parquet("data/ingatlan-listings-sorted.parquet")
listings_df = pl.read_parquet("data/ingatlan-listings-sorted.parquet")

In [41]:
%%timeit -c -r 10 -n 1
low_iqr_props(details_df, listings_df)

6.04 s ± 509 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [42]:
listings_df.flags

{'id': {'SORTED_ASC': False, 'SORTED_DESC': False},
 'day': {'SORTED_ASC': False, 'SORTED_DESC': False},
 'price': {'SORTED_ASC': False, 'SORTED_DESC': False}}

In [43]:
listings_df = listings_df.set_sorted(["id", "price"])

In [44]:
listings_df.flags

{'id': {'SORTED_ASC': True, 'SORTED_DESC': False},
 'day': {'SORTED_ASC': False, 'SORTED_DESC': False},
 'price': {'SORTED_ASC': False, 'SORTED_DESC': False}}

In [45]:
%%timeit -r 10 -n 1
low_iqr_props(details_df, listings_df)

569 ms ± 63.4 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


## Better solution

In [58]:
def low_iqr_props_optimal(
    details: pl.DataFrame, listings: pl.DataFrame, top_k: int = 20
) -> pl.DataFrame:
    return details.join(
        (
            listings.group_by("id")
            .agg(
                iqr=(pl.col("price").quantile(0.75) - pl.col("price").quantile(0.25)),
            )
            .select(pl.all().top_k_by("iqr", top_k, reverse=True))
        ),
        on="id",
        how="semi",
    )

In [59]:
listings_df = listings_df.sample(fraction=1.0, seed=42)

In [60]:
%%timeit -r 10 -n 1
low_iqr_props_optimal(details_df, listings_df)

1.19 s ± 147 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [61]:
listings_df.group_by("id").having(pl.len() > 100).mean()

id,day,price
i64,datetime[μs],f64
33822174,2024-05-07 15:00:00,370270.27027
34959015,2025-11-08 20:06:51.428571,208857.142857
34161677,2025-03-05 20:37:02.818792,203657.718121
34468013,2025-08-27 10:32:22.857142,889619.047619
7145991,2025-05-05 03:04:19.200,293360.0
32789011,2025-01-09 22:03:04.615384,169914.529915
34575091,2025-05-01 14:44:34.285714,180000.0
33995887,2024-04-10 09:22:54.545454,112681.818182
33765692,2024-11-19 04:09:38.102189,630264.90146


In [62]:
listings_df = listings_df.sort(
    "id",
    "price",
)

In [63]:
%%timeit -r 10 -n 1
low_iqr_props_optimal(details_df, listings_df)

533 ms ± 54.8 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [64]:
listings_df.flags

{'id': {'SORTED_ASC': True, 'SORTED_DESC': False},
 'day': {'SORTED_ASC': False, 'SORTED_DESC': False},
 'price': {'SORTED_ASC': False, 'SORTED_DESC': False}}

## Ultimate solution

In [65]:
def low_iqr_props_sigma(
    details: pl.DataFrame, listings: pl.DataFrame, top_k: int = 20
) -> pl.DataFrame:
    def _iqr(col: pl.Expr) -> pl.Expr:
        return col.quantile(0.75) - col.quantile(0.25)

    return details.join(
        (
            listings.group_by("id")
            .agg(
                iqr=(
                    pl.col("price")
                    .set_sorted()
                    .pipe(_iqr)
                )
            )
            .select(pl.all().top_k_by("iqr", top_k, reverse=True))
        ),
        on="id",
        how="semi",
    )


In [66]:
listings_df = listings_df.sample(fraction=1.0, seed=42)

In [67]:
%%timeit -r 10 -n 1
low_iqr_props_sigma(details_df, listings_df)

1.04 s ± 124 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [68]:
listings_df = listings_df.sort(
    "id",
    "price",
)

In [69]:
%%timeit -r 10 -n 1
low_iqr_props_sigma(details_df, listings_df)

947 ms ± 69.1 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
